In [ ]:
## Heatmap which is too complicated

from folium.plugins import HeatMapWithTime
from scipy.spatial import cKDTree

## convert dates into a specialized time-aware objects (datetime64)
gdf_fire["acq_date"] = pd.to_datetime(gdf_fire["acq_date"])


## sort days
unique_days = sorted(gdf_fire["acq_date"].dt.date.unique())

# gdf_fire["acq_date"]: access "aqc_date" column in gdf_fire -> returns Pandas-Series
# dt.: to access time/date values (from datetime)
# .date: removes hours from timeobject
# .unique: takes unique values (maybe unnecessary?)
# sorted(): sorts it in chronological order -> result is a sorted list




## create heatmap data, so HeatMapWithTime can use / work with it

data = [] #create empty list; here all the days will be saved
max_frp = gdf_fire["frp"].max()


for day in unique_days:
    subset = gdf_fire[gdf_fire["acq_date"].dt.date == day] ## ist das .dt.date nochmals nötig?

    coords = np.array([[row.geometry.y, row.geometry.x] for _, row in subset.iterrows()])

    # count for every point, how many points are within 2 degrees
    tree = cKDTree(coords)
    counts = np.array([len(tree.query_ball_point(p, r=2.0)) for p in coords])

    #norm it to 1
    max_count = counts.max() if counts.max() >0 else 1
    weights = counts / max_count

    day_data = [
        [float(coords[i][0]), float(coords[i][1]), float(weights[i])]
        for i in range(len(coords))
    ]

    data.append(day_data)

## create time index
time_index = [str(day) for day in unique_days]


## initialize basemap centered on South America, add a zoom control limit
min_lon, max_lon = -81.5, -35.0
min_lat, max_lat = -64.0, 15.5

base_map = folium.Map(
    max_bounds = True,
    location = [8, -69],
    zoom_start = 3,
    tiles = "CartoDB DarkMatter", #or better with CartoDB Positron
    control_scale = True,
    min_lat = min_lat,
    max_lat = max_lat,
    min_lon = min_lon,
    max_lon = max_lon

)

## heatmap
# 0-1 as threshold values for pixel intensity
HeatMapWithTime(
    data,
    index=time_index,
    auto_play = True,
    radius = 10,
    min_opacity= 0.3,
    gradient = {
        0.0: "#0000ff", # isolated value
        0.5: "#f9e107", #medium density
        1.0: "#ff0101", # high density
    }
).add_to(base_map)


## html legend
legend_html = """
<div style = "position: fixed;
bottom: 50px; 
right: 50px;
width: 170px; 
height: 100px;
background-color: lightgrey;
z-index:9999;
font-size:14px;
padding: 10px;
border: 2px solid grey;
">

<b>Fire Density</b><br>
<i style="background:#0000ff;width:20px;height:10px;float:left;margin-right:8px;"></i> Low Density<br>

<i style="background:#f9e107;width:20px;height:10px;float:left;margin-right:8px;"></i> Medium Density<br>

<i style="background:#ff0101;width:20px;height:10px;float:left;margin-right:8px;"></i> High Density

</div>
"""

## add legend to the map
base_map.get_root().html.add_child(folium.Element(legend_html))


## save map because I cannot open it in VS Code directly
base_map.save("Time_Animated_Heatmap2.html")

base_map
